# Model 2 — Next-MOT prediction with the UK data: history, true odometer and the Model-1 estimate  (experiments only)

Same question as notebook 08 (Dutch data) on an independent country that **also has the odometer**: given everything known at one MOT of a car, does the **next annual MOT** (about a year later) find a defect in brakes, electrical or engine-related items?
Because the UK data carry the odometer, this notebook can answer the question the supervisor raised: **in Israel mileage would be estimated by Model 1; what does that cost against a true odometer, now in a forward-looking setting?**

**Data.** DVSA MOT, class-4 cars, normal tests. A pseudo-random **1-in-41 subset of vehicles** (not of tests) was sampled from the 2022 and 2023 files (`ml/src/mot_vehicle_sample.py`), so each sampled vehicle keeps all its tests and can be followed from 2022 into 2023.
As in notebook 04, the defect-item stream is only readable for the first part of each year (decoder limit; also the sampler stops reading early), so only tests before the detected cut-off date of each year are used. A test without a recorded item is a test without a recorded defect only inside that window.

**Pairs.** Test *i* (2022 window) -> the same vehicle's next test *i+1* (2023 window), 6 to 20 months later. Features: only what is known at test *i*. Target: defect by system at test *i+1*.

**Leakage and validation.**
* Features from test *i* only; the target is test *i+1*.
* Temporal split: training pairs have their target test early in the 2023 window, test pairs have it later; vehicles are disjoint across train / validation / test.
* The Stage-1 (Model 1 style) mileage estimate is trained on **different vehicles** than the ones in Stage 2 (honest two-stage setup).
* Three random vehicle splits; spread reported.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.options.display.width = 190; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from real_outcome_data import load_uk_veh
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from xgboost import XGBClassifier, XGBRegressor
try:
    display
except NameError:
    display = print
a, cut_a = load_uk_veh(2022); b, cut_b = load_uk_veh(2023)
print(f"2022: {len(a):,} tests before {cut_a.date()} | 2023: {len(b):,} tests before {cut_b.date()}")
SYS = ["brakes", "electrical", "engine_emissions"]
d = pd.concat([a, b]).sort_values(["unit", "date"]).reset_index(drop=True)
g = d.groupby("unit")
d["next_date"] = g.date.shift(-1)
for s in SYS + ["any_defect"]:
    d["cur_" + s] = d[s]; d["next_" + s] = g[s].shift(-1)
d["gap_next"] = (d.next_date - d.date).dt.days / 365.25
pairs = d[d.next_date.notna() & d.gap_next.between(0.5, 1.7) & (d.date.dt.year == 2022) & (d.next_date.dt.year == 2023)].copy()
pairs["target_date"] = pairs.next_date
print(f"vehicles tests {len(d):,}; pairs (2022 -> 2023) {len(pairs):,} of {pairs.unit.nunique():,} vehicles; gap (years) median {pairs.gap_next.median():.2f}")
print("target dates:", pairs.target_date.dt.to_period('M').value_counts().sort_index().to_dict())
pairs = pairs[pairs.fuel.isin(["petrol", "diesel", "hybrid", "electric"]) & pairs.age_year.between(3, 25)]
pairs["make_c"] = pairs.make.where(pairs.make.isin(pairs.make.value_counts().head(60).index), "OTHER")
pairs["model_c"] = pairs.model.where(pairs.model.isin(pairs.model.value_counts().head(400).index), "OTHER")
CATS = {c: sorted(pairs[c].unique()) for c in ["fuel", "make_c", "model_c"]}
print("prevalence of the target at test i+1:", {s: round(float(pairs['next_' + s].mean()), 4) for s in SYS})
for s in SYS:
    print(f"  {s}: defect at i+1 given defect at i = {pairs.loc[pairs['cur_' + s] == 1, 'next_' + s].mean():.3f} | given none at i = {pairs.loc[pairs['cur_' + s] == 0, 'next_' + s].mean():.3f}")

2022: 374,238 tests before 2022-07-03 | 2023: 368,402 tests before 2023-06-26


vehicles tests 742,640; pairs (2022 -> 2023) 301,886 of 301,886 vehicles; gap (years) median 1.00
target dates: {Period('2023-01', 'M'): 54652, Period('2023-02', 'M'): 55027, Period('2023-03', 'M'): 71571, Period('2023-04', 'M'): 38984, Period('2023-05', 'M'): 43820, Period('2023-06', 'M'): 37832}


prevalence of the target at test i+1: {'brakes': 0.0677, 'electrical': 0.115, 'engine_emissions': 0.0302}
  brakes: defect at i+1 given defect at i = 0.162 | given none at i = 0.062
  electrical: defect at i+1 given defect at i = 0.263 | given none at i = 0.099
  engine_emissions: defect at i+1 given defect at i = 0.218 | given none at i = 0.026


## Variants
| Variant | Features known at test *i* |
| --- | --- |
| **U0** | age, fuel, make, model, planned interval to the next test |
| **U1** | U0 + results of all systems at test *i* (the car's own recent history) |
| **U2** | U1 + **Model-1 style estimated odometer** (Stage 1 trained on other vehicles from registry fields only) — the Israeli scenario |
| **U3** | U1 + **true odometer** at test *i* |
| **U4** | U1 + true odometer and the estimate's gap to it (usage above or below what the registry fields predict) |

In [2]:
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop"); t = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum())
def capture(y, p, frac=0.2):
    k = int(len(p) * frac); idx = np.argsort(-p)[:k]; y = np.asarray(y); return float(y[idx].sum() / y.sum())
def best_f1_thr(y, p):
    pr, rc, th = precision_recall_curve(y, p); f1 = 2 * pr * rc / np.clip(pr + rc, 1e-12, None); return float(th[int(np.nanargmax(f1[:-1]))])
def prep(df, cols):
    X = df[cols].copy()
    for c in cols:
        if c in CATS: X[c] = pd.Categorical(X[c], categories=CATS[c])
    return X
def fit_xgb(X, y):
    return XGBClassifier(n_estimators=250, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(X, y)
BASE = ["age_year", "gap_next", "fuel", "make_c", "model_c"]; HIST = ["cur_brakes", "cur_electrical", "cur_engine_emissions", "cur_any_defect"]
VARS = {"U0": BASE, "U1": BASE + HIST, "U2": BASE + HIST + ["km_est"], "U3": BASE + HIST + ["km"], "U4": BASE + HIST + ["km", "km_gap"]}
def run(seed):
    veh = np.array(pairs.unit.drop_duplicates().tolist()); rng = np.random.default_rng(seed); rng.shuffle(veh); n = len(veh)
    s1v, trv, vav, tev = set(veh[:int(.25 * n)]), set(veh[int(.25 * n):int(.6 * n)]), set(veh[int(.6 * n):int(.75 * n)]), set(veh[int(.75 * n):])
    # Stage 1 learns the odometer from registry fields on vehicles that Stage 2 never sees (uses odometers only, no defect labels)
    s1 = pairs[pairs.unit.isin(s1v)]
    reg = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(prep(s1, ["age_year", "fuel", "make_c"]), s1.km)
    P = pairs[~pairs.unit.isin(s1v)].copy(); P["km_est"] = reg.predict(prep(P, ["age_year", "fuel", "make_c"])); P["km_gap"] = P.km - P.km_est
    tmid = P.target_date.quantile(0.6)
    tr, va, te = P[P.unit.isin(trv) & (P.target_date <= tmid)], P[P.unit.isin(vav) & (P.target_date <= tmid)], P[P.unit.isin(tev) & (P.target_date > tmid)]
    assert not (set(tr.unit) & set(te.unit)) and tr.target_date.max() <= tmid < te.target_date.min()
    print(f"seed {seed}: train {len(tr):,} | validation {len(va):,} | test {len(te):,} pairs; test index checksum {int(te.index.to_numpy().sum())}")
    out = []
    for s in SYS:
        t = "next_" + s; y = te[t].to_numpy()
        for v, cols in VARS.items():
            m = fit_xgb(prep(tr, cols), tr[t]); p = m.predict_proba(prep(te, cols))[:, 1]; pv = m.predict_proba(prep(va, cols))[:, 1]
            thr = best_f1_thr(va[t], pv); pred = p >= thr; tp = int((pred & (y == 1)).sum())
            out.append({"seed": seed, "system": s, "variant": v, "model": "XGBoost", "n test": len(te), "prevalence": y.mean(), "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
                        "capture 20%": capture(y, p), "ECE": ece(y, p), "flagged": pred.mean(), "Precision": tp / max(pred.sum(), 1), "Recall": tp / y.sum(), "F1": f1_score(y, pred)})
        cols = [c for c in VARS["U1"]]
        num = [c for c in cols if c not in ("fuel", "make_c", "model_c", "age_year")]
        lr = Pipeline([("p", ColumnTransformer([("a", SplineTransformer(n_knots=6, degree=3), ["age_year"]), ("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False), ["fuel", "make_c"])])), ("m", LogisticRegression(max_iter=2000))])
        lr.fit(tr[cols].astype({"fuel": str, "make_c": str}), tr[t]); p = lr.predict_proba(te[cols].astype({"fuel": str, "make_c": str}))[:, 1]; pv = lr.predict_proba(va[cols].astype({"fuel": str, "make_c": str}))[:, 1]
        thr = best_f1_thr(va[t], pv); pred = p >= thr; tp = int((pred & (y == 1)).sum())
        out.append({"seed": seed, "system": s, "variant": "U1", "model": "Logistic Regression", "n test": len(te), "prevalence": y.mean(), "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "capture 20%": capture(y, p),
                    "ECE": ece(y, p), "flagged": pred.mean(), "Precision": tp / max(pred.sum(), 1), "Recall": tp / y.sum(), "F1": f1_score(y, pred)})
        prev = y.mean(); out.append({"seed": seed, "system": s, "variant": "flag every car", "model": "baseline", "n test": len(te), "prevalence": prev, "ROC-AUC": 0.5, "PR-AUC": prev, "capture 20%": 0.2, "ECE": np.nan, "flagged": 1.0, "Precision": prev, "Recall": 1.0, "F1": 2 * prev / (1 + prev)})
    r2 = r2_score(te.km, te.km_est)
    return out, r2, len(tr), len(te)
res, r2s = [], []
for seed in [0, 1, 2]:
    o, r2, ntr, nte = run(seed); res += o; r2s.append(r2)
R = pd.DataFrame(res)
print(f"train pairs ~{ntr:,}, test pairs ~{nte:,} per split; Stage-1 R2 on the test vehicles: {np.mean(r2s):.3f}")

seed 0: train 63,042 | validation 27,127 | test 29,914 pairs; test index checksum 11017866386


seed 1: train 63,037 | validation 26,992 | test 29,724 pairs; test index checksum 11049438600


seed 2: train 62,993 | validation 27,216 | test 29,945 pairs; test index checksum 11139799666


train pairs ~62,993, test pairs ~29,945 per split; Stage-1 R2 on the test vehicles: 0.508


## Results (mean of 3 vehicle splits)

In [3]:
agg = R.groupby(["system", "variant", "model"], sort=False).agg(**{"ROC-AUC": ("ROC-AUC", "mean"), "AUC spread": ("ROC-AUC", lambda x: x.max() - x.min()), "PR-AUC": ("PR-AUC", "mean"), "prevalence": ("prevalence", "mean"),
                                                                   "capture 20%": ("capture 20%", "mean"), "ECE": ("ECE", "mean"), "Precision": ("Precision", "mean"), "Recall": ("Recall", "mean"), "F1": ("F1", "mean")}).reset_index()
display(agg)
print("per-seed ROC-AUC (XGBoost):"); display(R[R.model == "XGBoost"].pivot_table(index=["system", "variant"], columns="seed", values="ROC-AUC"))
pv = R[R.model == "XGBoost"].pivot_table(index="system", columns="variant", values="ROC-AUC")[list(VARS)]
pv["U1 - U0 (history)"] = pv["U1"] - pv["U0"]; pv["U2 - U1 (Model-1 estimate)"] = pv["U2"] - pv["U1"]; pv["U3 - U1 (true odometer)"] = pv["U3"] - pv["U1"]; pv["U3 - U2 (true vs estimated)"] = pv["U3"] - pv["U2"]
display(pv)

,system,variant,model,ROC-AUC,AUC spread,PR-AUC,prevalence,capture 20%,ECE,Precision,Recall,F1
0,brakes,U0,XGBoost,0.6691,0.0043,0.1226,0.0675,0.3895,0.0100,0.1293,0.4228,0.1966
1,brakes,U1,XGBoost,0.6790,0.0049,0.1345,0.0675,0.4082,0.0107,0.1339,0.4378,0.2038
2,brakes,U2,XGBoost,0.6799,0.0052,0.1352,0.0675,0.4101,0.0103,0.1351,0.4459,0.2068
3,brakes,U3,XGBoost,0.6931,0.0116,0.1412,0.0675,0.4232,0.0100,0.1476,0.3877,0.2124
4,brakes,U4,XGBoost,0.6930,0.0109,0.1407,0.0675,0.4257,0.0103,0.1460,0.4101,0.2152
5,brakes,U1,Logistic Regression,0.6894,0.0115,0.1413,0.0675,0.4126,0.0039,0.1377,0.4218,0.2072
6,brakes,flag every car,baseline,0.5000,0.0000,0.0675,0.0675,0.2000,NaN,0.0675,1.0000,0.1264
7,electrical,U0,XGBoost,0.7117,0.0047,0.2191,0.1147,0.4135,0.0115,0.2140,0.5584,0.3094
8,electrical,U1,XGBoost,0.7218,0.0032,0.2368,0.1147,0.4264,0.0124,0.2255,0.5476,0.3194
9,electrical,U2,XGBoost,0.7217,0.0030,0.2372,0.1147,0.4286,0.0124,0.2224,0.5550,0.3174


per-seed ROC-AUC (XGBoost):


seed                          0      1      2
system           variant                     
brakes           U0      0.6707 0.6664 0.6702
                 U1      0.6774 0.6774 0.6822
                 U2      0.6777 0.6792 0.6829
                 U3      0.6885 0.6907 0.7001
                 U4      0.6887 0.6909 0.6995
electrical       U0      0.7125 0.7089 0.7136
                 U1      0.7205 0.7213 0.7237
                 U2      0.7205 0.7211 0.7235
                 U3      0.7346 0.7328 0.7365
                 U4      0.7349 0.7319 0.7376
engine_emissions U0      0.7073 0.7122 0.7100
                 U1      0.7295 0.7463 0.7397
                 U2      0.7284 0.7453 0.7387
                 U3      0.7485 0.7594 0.7511
                 U4      0.7490 0.7600 0.7538

variant,U0,U1,U2,U3,U4,U1 - U0 (history),U2 - U1 (Model-1 estimate),U3 - U1 (true odometer),U3 - U2 (true vs estimated)
system,,,,,,,,,
brakes,0.6691,0.6790,0.6799,0.6931,0.6930,0.0099,0.0009,0.0141,0.0132
electrical,0.7117,0.7218,0.7217,0.7346,0.7348,0.0102,-0.0001,0.0128,0.0129
engine_emissions,0.7098,0.7385,0.7375,0.7530,0.7542,0.0287,-0.0010,0.0145,0.0156
